In [1]:
!pip install -q -U transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 12.1 MB/s eta 0:00:00


In [2]:
import torch
import os

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel
)

In [3]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. Enable GPU in Colab.")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


SELECT MODEL

In [4]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print("Model:", MODEL_NAME)

Model: Qwen/Qwen2.5-0.5B-Instruct


CREATE TRAINING DATASET

In [5]:
data = [
    {
        "instruction": "What is RAG?",
        "response": "RAG stands for Retrieval-Augmented Generation. It retrieves relevant information from external documents and gives that information to an LLM so the model can generate a better answer."
    },

    {
        "instruction": "What is LoRA?",
        "response": "LoRA stands for Low-Rank Adaptation. It is a parameter-efficient fine-tuning method that freezes the original model and trains small low-rank matrices."
    },

    {
        "instruction": "What is QLoRA?",
        "response": "QLoRA combines 4-bit quantization with LoRA. The base model is loaded in low precision and kept frozen while small LoRA adapters are trained."
    },

    {
        "instruction": "What is an LLM?",
        "response": "An LLM is a Large Language Model trained on a large amount of text to understand and generate human language."
    },

    {
        "instruction": "What is fine-tuning?",
        "response": "Fine-tuning means taking a pretrained model and training it further on a specific dataset so it becomes better at a particular task or behavior."
    },

    {
        "instruction": "What is a transformer?",
        "response": "A Transformer is a neural network architecture based mainly on attention mechanisms. Modern language models such as GPT use Transformer architectures."
    },

    {
        "instruction": "What is self-attention?",
        "response": "Self-attention allows each token in a sequence to look at other tokens and determine which ones are important for understanding the current token."
    },

    {
        "instruction": "What is a token?",
        "response": "A token is a small unit of text processed by a language model. Depending on the tokenizer, a token can be a word, part of a word, punctuation, or another text unit."
    },

    {
        "instruction": "What is an embedding?",
        "response": "An embedding is a numerical vector representation of data such as a token, sentence, or document."
    },

    {
        "instruction": "What is a vector database?",
        "response": "A vector database stores vector representations and allows similarity searches between vectors."
    },

    {
        "instruction": "What is prompt engineering?",
        "response": "Prompt engineering is the process of designing instructions and inputs that guide an AI model toward the desired output."
    },

    {
        "instruction": "What is an AI agent?",
        "response": "An AI agent is a system that can reason about a task, use tools, access information, and perform actions to achieve a goal."
    }
]

dataset = Dataset.from_list(data)

print(dataset)

Dataset({
    features: ['instruction', 'response'],
    num_rows: 12
})


In [6]:
print(dataset[0])

{'instruction': 'What is RAG?', 'response': 'RAG stands for Retrieval-Augmented Generation. It retrieves relevant information from external documents and gives that information to an LLM so the model can generate a better answer.'}


TRAIN\TEST SPLIT

In [7]:
dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

print("Training examples:", len(train_dataset))
print("Testing examples:", len(test_dataset))

Training examples: 9
Testing examples: 3


LOAD TOKENIZER

In [8]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded.")
print("Vocabulary size:", len(tokenizer))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded.
Vocabulary size: 151665


PREPARE TRAINING TEXT

In [9]:
def format_example(example):

    text = (
        "### Instruction:\n"
        + example["instruction"]
        + "\n\n"
        + "### Response:\n"
        + example["response"]
        + tokenizer.eos_token
    )

    return {
        "text": text
    }


train_dataset = train_dataset.map(format_example)
test_dataset = test_dataset.map(format_example)

print(train_dataset[0]["text"])

Map:   0%|          | 0/9 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

### Instruction:
What is a vector database?

### Response:
A vector database stores vector representations and allows similarity searches between vectors.<|im_end|>


TOKENIZATION

In [22]:
MAX_LENGTH = 256

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )


tokenized_train = train_dataset.map(
    tokenize_function,
    batched=False,
    remove_columns=train_dataset.column_names
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=False,
    remove_columns=test_dataset.column_names
)

print(tokenized_train.column_names)

Map:   0%|          | 0/9 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

['input_ids', 'attention_mask']


4-BIT QLORA CONFIGURATION

In [23]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,

    # QLoRA commonly uses NF4
    bnb_4bit_quant_type="nf4",

    # Double quantization saves additional memory
    bnb_4bit_use_double_quant=True,

    # Computation will use FP16
    bnb_4bit_compute_dtype=torch.float16
)

print("4-bit quantization configured.")

4-bit quantization configured.


LOAD MODEL IN 4 BITS

In [24]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded in 4-bit.")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded in 4-bit.


PREPARE MODEL FOR K -BIT TRAINING

In [25]:
model = prepare_model_for_kbit_training(model)

print("Model prepared for k-bit training.")

Model prepared for k-bit training.


CREATE LORE CONFIGURATION

In [26]:
lora_config = LoraConfig(
    r=8,

    lora_alpha=16,

    lora_dropout=0.05,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],

    bias="none",

    task_type="CAUSAL_LM"
)

In [27]:
model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


DATA COLLECTOR

In [28]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False   #means we are training a causal language model, like GPT
)

TRAINING ARGUMENTS

In [29]:
training_args = TrainingArguments(
    output_dir="./qlora_output",

    num_train_epochs=5,

    per_device_train_batch_size=2,

    per_device_eval_batch_size=2,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    logging_steps=1,

    save_strategy="epoch",

    eval_strategy="epoch",

    fp16=True,

    report_to="none",

    remove_unused_columns=False
)

CREATE TRAINER

In [30]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator
)

In [31]:
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,3.192507,3.136761
2,3.123417,3.062743
3,2.724897,2.924809
4,3.150795,2.837067
5,3.019030,2.796299


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=10, training_loss=3.2556222677230835, metrics={'train_runtime': 15.5543, 'train_samples_per_second': 2.893, 'train_steps_per_second': 0.643, 'total_flos': 3883439755008.0, 'train_loss': 3.2556222677230835, 'epoch': 5.0})

EVALUATE

In [32]:
results = trainer.evaluate()

print(results)

Training Loss,Validation Loss,Epoch
3.019030,2.796299,5


{'eval_loss': 2.7962987422943115}


SAVE

In [33]:
ADAPTER_PATH = "./my_qlora_adapter"

model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

print("QLoRA adapter saved to:", ADAPTER_PATH)

QLoRA adapter saved to: ./my_qlora_adapter


LOAD

In [34]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [35]:
trained_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

trained_model.eval()

print("QLoRA adapter loaded.")

QLoRA adapter loaded.


TEST THE MODEL

In [36]:
prompt = """### Instruction:
What is QLoRA?

### Response:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

inputs = {
    key: value.to(trained_model.device)
    for key, value in inputs.items()
}

with torch.no_grad():

    outputs = trained_model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True,
        top_p=0.9
    )

generated_text = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(generated_text)

### Instruction:
What is QLoRA?

### Response:
QLoRA stands for QiLlama, which means "Qi" (the Chinese word for "qubit") and "Llama", a model used in deep learning. It was created by Qwen.AI to enable users to use the Qwen AI models in their own projects. Here are some key points about QLoRA:

1. **QLoRA is a new generation of Qwen AI models**: QLoRA uses a new architecture called Q-Llama, which includes multiple layers


                    DATASET
                       │
                       ▼
                  TOKENIZER
                       │
                       ▼
              ┌─────────────────┐
              │  Qwen Base LLM  │
              └─────────────────┘
                       │
                 4-bit NF4
                       │
                       ▼
             Frozen Base Weights
                       │
                       │
              ┌────────┴────────┐
              │                 │
              ▼                 ▼
          Q/K/V/O             LoRA
         projections       A + B matrices
              │                 │
              │           Trainable
              │                 │
              └────────┬────────┘
                       ▼
                 Transformer
                       │
                       ▼
                    Logits
                       │
                       ▼
                     Loss
                       │
                       ▼
                 Backpropagation
                       │
                       ▼
              Update LoRA only